# 07. Cohort Rating Packet Export

Local-only entry point for the configured reference-complete test cohort. Builds blinded assessment and feedback packets for G1/G2/G3 plus one teacher-reference packet per document. No model or tracing calls are made.

For the selected 16-document cohort, expect 48 assessment-quality packets, 48 feedback-quality packets, and 16 reference packets. Only counts are displayed; protected text and the condition key stay in approved storage.

In [11]:
import importlib
import json
import os
from collections import Counter
from pathlib import Path

from dotenv import load_dotenv
import reflection_assessment_feedback.rating_packets as rating_packets_module
from reflection_assessment_feedback.experiment_config import load_experiment_config

rating_packets_module = importlib.reload(rating_packets_module)
build_development_rating_packet_bundle = rating_packets_module.build_development_rating_packet_bundle

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name != "prebi_v1":
    raise RuntimeError("Run this notebook from the prebi_v1 project root.")
load_dotenv(PROJECT_ROOT / ".env", override=False)
root_setting = os.environ.get("PREBI_DATA_ROOT")
if not root_setting:
    raise RuntimeError("Set PREBI_DATA_ROOT in the local .env or environment.")
PROTECTED_ROOT = Path(root_setting).expanduser().resolve()
DATASET_CONFIG = load_experiment_config()["dataset"]
COHORT_ID = DATASET_CONFIG["generation_cohort"]
COHORT_FILE_PREFIX = COHORT_ID.lower().replace("_", "-")
WRITE_PACKETS = True
OUTPUT_DIRECTORY = PROTECTED_ROOT / "rating-packets" / "development"
PACKET_PATH = OUTPUT_DIRECTORY / f"{COHORT_FILE_PREFIX}-rating-packets.json"
KEY_PATH = OUTPUT_DIRECTORY / f"{COHORT_FILE_PREFIX}-condition-key.json"
print(f"Local setup ready for {COHORT_ID}. Packet export enabled: {WRITE_PACKETS}.")

Local setup ready for remaining-test-reference-complete-v1. Packet export enabled: True.


In [12]:
packet_bundle, condition_key = build_development_rating_packet_bundle(PROTECTED_ROOT, write=False)
packets = packet_bundle["packets"]
cohort_document_count = sum(packet["task"] == "feedback_implied_score" for packet in packets)
expected_counts = {
    "assessment_quality": cohort_document_count * 3,
    "feedback_quality": cohort_document_count * 3,
    "feedback_implied_score": cohort_document_count,
}
task_counts = Counter(packet["task"] for packet in packets)
if task_counts != expected_counts:
    raise ValueError("Packet task counts differ from the selected cohort size.")
packet_ids = {packet["packet_id"] for packet in packets}
expected_packet_count = cohort_document_count * 7
if len(packet_ids) != expected_packet_count or len(condition_key["records"]) != expected_packet_count:
    raise ValueError("Packet or key identifiers are missing or duplicated.")
if packet_ids != {record["packet_id"] for record in condition_key["records"]}:
    raise ValueError("Packet bundle and protected key do not match.")
for packet in packets:
    if packet["condition_blinded"] is not True:
        raise ValueError("An unblinded packet was returned.")
    if {"run_id", "document_id", "condition", "model_name", "analysis", "gold"}.intersection(packet):
        raise ValueError("A packet contains protected provenance fields.")
print(f"Local preflight passed for {cohort_document_count} documents. No files written.")
print(dict(task_counts))

Local preflight passed for 16 documents. No files written.
{'assessment_quality': 48, 'feedback_quality': 48, 'feedback_implied_score': 16}


## Export and verify

The cohort-specific filenames are derived from the configured cohort ID. The builder refuses to overwrite existing packet or key exports. The verification cell reads both files back and compares them with the written snapshots. Keep the condition key separately restricted; never upload it with the blinded packet bundle.

In [13]:
if not WRITE_PACKETS:
    print("Export disabled. No files written. Set WRITE_PACKETS=True in Cell 2 to export deliberately.")
else:
    packet_bundle, condition_key = build_development_rating_packet_bundle(PROTECTED_ROOT, write=True)
    saved_bundle = json.loads(PACKET_PATH.read_text(encoding="utf-8"))
    saved_key = json.loads(KEY_PATH.read_text(encoding="utf-8"))
    if saved_bundle != packet_bundle or saved_key != condition_key:
        raise ValueError("Saved exports differ from the written snapshots.")
    print("Export completed and read-back verified.")
    print(dict(Counter(packet["task"] for packet in saved_bundle["packets"])))
    print(f"Upload only this packet bundle: {PACKET_PATH}")
    print("Condition key saved separately in protected storage. Never upload it to the rating UI.")

Export completed and read-back verified.
{'assessment_quality': 48, 'feedback_quality': 48, 'feedback_implied_score': 16}
Upload only this packet bundle: /Users/quynguyen/Documents/CodingProjects/prebi_data/rating-packets/development/remaining-test-reference-complete-v1-rating-packets.json
Condition key saved separately in protected storage. Never upload it to the rating UI.
